## MISA (2024-2025)
- Alohan'ny mamerina dia avereno atao Run ny notebook iray manontolo. Ny fanaovana azy dia redémarrena mihitsy ny kernel aloha (jereo menubar, safidio **Kernel$\rightarrow$Restart Kernel and Run All Cells**).

- Izay misy hoe `YOUR CODE HERE` na `YOUR ANSWER HERE` ihany no fenoina. Afaka manampy cells vaovao raha ilaina. Aza adino ny mameno references eo ambany raha ilaina.

## References
ChatGpt, ClaudeAI

---

In [1]:
from random import randrange
import numpy as np
from sklearn.metrics import mean_squared_error, log_loss
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import normalize


def grad_check_sparse(f, x, analytic_grad, num_checks=10, h=1e-5, error=1e-9):
    """
    sample a few random elements and only return numerical
    in this dimensions
    """

    for i in range(num_checks):
        ix = tuple([randrange(m) for m in x.shape])

        oldval = x[ix]
        x[ix] = oldval + h  # increment by h
        fxph = f(x)  # evaluate f(x + h)
        x[ix] = oldval - h  # increment by h
        fxmh = f(x)  # evaluate f(x - h)
        x[ix] = oldval  # reset

        grad_numerical = (fxph - fxmh) / (2 * h)
        grad_analytic = analytic_grad[ix]
        rel_error = abs(grad_numerical - grad_analytic) / (
            abs(grad_numerical) + abs(grad_analytic)
        )
        print(
            "numerical: %f analytic: %f, relative error: %e"
            % (grad_numerical, grad_analytic, rel_error)
        )
        assert rel_error < error

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [2]:
data = load_diabetes()
X, y = data.data, data.target

In [3]:
def mse_loss_vectorized(w, b, X, y):
    """
    MSE loss function WITHOUT FOR LOOPs , NO REGULARIZATION
    
    Returns a tuple of:
    - loss 
    - gradient with respect to weights w
    - gradient with respect to bias b
    """
    loss = 0.0
    dw = np.zeros_like(w)
    
    N = X.shape[0]
    y_pred = X @ w + b
    residuals = y_pred - y
    loss = np.sum(residuals ** 2) / N
    dw = 2 * X.T @ residuals / N
    db = 2 * np.sum(residuals) / N
    
    return loss, dw, np.array(db).reshape(1,)

In [4]:
def soft_threshold(x, delta):
    return np.sign(x) * np.maximum(np.abs(x) - delta, 0)

# Lasso Subgradient Descent

In [5]:
def l1_subgradient(w):
    """
    Subgradient of the L1 loss
    """
    dw = np.zeros_like(w)
    dw = np.sign(w)
    return dw
    

def lasso_subgradient_mse_loss_vectorized(w, b, X, y, alpha):
    """
    MSE loss function adding the subgradient for w
    """
    loss, dw, db = mse_loss_vectorized(w, b, X, y)
    # Add the subgradient to dw
    dw = dw + alpha * l1_subgradient(w)
    return loss, dw, db

In [6]:
class LassolSubgradientDescent():
    def __init__(self,  alpha=0.1):
        self.w = None
        self.b = None
        self.alpha = alpha

    def train(self, X, y, learning_rate=1e-3, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w is None: # Initialization
            self.w = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            indices = np.random.choice(N, batch_size, replace=True)
            X_batch = X[indices]
            y_batch = y[indices]
            
            loss, dw, db = self.loss(X_batch, y_batch)

            self.w -= learning_rate * dw
            self.b -= learning_rate * db
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))
    
    
    def predict(self, X):
        return X @ self.w + self.b

    def loss(self, X_batch, y_batch):
        return lasso_subgradient_mse_loss_vectorized(self.w, self.b, X_batch, y_batch, self.alpha)

In [7]:
model = LassolSubgradientDescent(alpha=0.1)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse = mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 25978.879109
iteration 10000 / 200000: loss 3199.225685
iteration 20000 / 200000: loss 2663.229799
iteration 30000 / 200000: loss 2939.524563
iteration 40000 / 200000: loss 2475.329843
iteration 50000 / 200000: loss 2553.158332
iteration 60000 / 200000: loss 2526.100053
iteration 70000 / 200000: loss 2352.181155
iteration 80000 / 200000: loss 3088.023228
iteration 90000 / 200000: loss 3123.250783
iteration 100000 / 200000: loss 2718.988519
iteration 110000 / 200000: loss 2527.366911
iteration 120000 / 200000: loss 2918.320664
iteration 130000 / 200000: loss 3154.214902
iteration 140000 / 200000: loss 2809.868710
iteration 150000 / 200000: loss 3026.182100
iteration 160000 / 200000: loss 3401.012899
iteration 170000 / 200000: loss 2964.721808
iteration 180000 / 200000: loss 2738.034714
iteration 190000 / 200000: loss 2952.176712
MSE scikit-learn: 2912.52749260362
MSE Coordinate descent model : 2889.388766742093


In [8]:
model = LassolSubgradientDescent(alpha=2)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse = mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 30026.262797
iteration 10000 / 200000: loss 4589.863403
iteration 20000 / 200000: loss 3409.968092
iteration 30000 / 200000: loss 4480.565684
iteration 40000 / 200000: loss 3741.414175
iteration 50000 / 200000: loss 3962.695315
iteration 60000 / 200000: loss 3351.716954
iteration 70000 / 200000: loss 4149.426846
iteration 80000 / 200000: loss 4173.387110
iteration 90000 / 200000: loss 3810.988501
iteration 100000 / 200000: loss 3950.326236
iteration 110000 / 200000: loss 3878.426359
iteration 120000 / 200000: loss 3669.531970
iteration 130000 / 200000: loss 3603.804373
iteration 140000 / 200000: loss 3880.315327
iteration 150000 / 200000: loss 3710.775702
iteration 160000 / 200000: loss 3693.493030
iteration 170000 / 200000: loss 3569.247078
iteration 180000 / 200000: loss 3775.579235
iteration 190000 / 200000: loss 3448.655080
MSE scikit-learn: 5650.290772564547
MSE Coordinate descent model : 3810.3527144345444


# Lasso Proximal Gradient Descent (iterative soft thresholding algorithm or ISTA)

In [9]:
class LassoProximalGradientDescent():
    def __init__(self,  alpha=0.1):
        self.w = None
        self.b = None
        self.alpha = alpha

    def train(self, X, y, learning_rate=1e-3, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w is None: # Initialization
            self.w = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            indices = np.random.choice(N, batch_size, replace=True)
            X_batch = X[indices]
            y_batch = y[indices]
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.  
            # PROJECT THE GRADIENT FOR w USING soft_threshold
            self.w = soft_threshold(self.w - learning_rate * dw, learning_rate * self.alpha)
            self.b -= learning_rate * db
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))

    def predict(self, X):
        return X @ self.w + self.b

    def loss(self, X_batch, y_batch):
        return mse_loss_vectorized(self.w, self.b, X_batch, y_batch)

In [10]:
model = LassoProximalGradientDescent(alpha=0.1)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 29958.185588
iteration 10000 / 200000: loss 3358.595992
iteration 20000 / 200000: loss 2585.077568
iteration 30000 / 200000: loss 2639.279658
iteration 40000 / 200000: loss 2611.462906
iteration 50000 / 200000: loss 2965.866381
iteration 60000 / 200000: loss 3022.112479
iteration 70000 / 200000: loss 3129.186925
iteration 80000 / 200000: loss 3466.997700
iteration 90000 / 200000: loss 2604.043882
iteration 100000 / 200000: loss 2836.025285
iteration 110000 / 200000: loss 2946.274665
iteration 120000 / 200000: loss 2838.287752
iteration 130000 / 200000: loss 2631.517052
iteration 140000 / 200000: loss 2672.299541
iteration 150000 / 200000: loss 2681.288438
iteration 160000 / 200000: loss 2792.478628
iteration 170000 / 200000: loss 2691.487089
iteration 180000 / 200000: loss 2751.205822
iteration 190000 / 200000: loss 3368.533043
MSE scikit-learn: 2912.52749260362
MSE Coordinate descent model : 2888.6435752394245


In [11]:
model = LassoProximalGradientDescent(alpha=2)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 28937.142978
iteration 10000 / 200000: loss 4326.534715
iteration 20000 / 200000: loss 4036.520175
iteration 30000 / 200000: loss 4215.318020
iteration 40000 / 200000: loss 3960.560897
iteration 50000 / 200000: loss 3416.931498
iteration 60000 / 200000: loss 3621.947731
iteration 70000 / 200000: loss 3661.964041
iteration 80000 / 200000: loss 4387.643324
iteration 90000 / 200000: loss 4034.923647
iteration 100000 / 200000: loss 3612.259386
iteration 110000 / 200000: loss 3439.942914
iteration 120000 / 200000: loss 3877.591206
iteration 130000 / 200000: loss 3708.004932
iteration 140000 / 200000: loss 3588.634108
iteration 150000 / 200000: loss 3778.648780
iteration 160000 / 200000: loss 3801.578550
iteration 170000 / 200000: loss 3457.682844
iteration 180000 / 200000: loss 3702.360625
iteration 190000 / 200000: loss 4211.738286
MSE scikit-learn: 5650.290772564547
MSE Coordinate descent model : 3810.250131345866


# Lasso Projected Gradient Descent

In [12]:
class LassoProjectedGradientDescent():
    def __init__(self,  alpha=0.1):
        self.w_p = None
        self.w_n = None
        self.b = None
        self.alpha = alpha

    def train(self, X, y, learning_rate=1e-3, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w_p is None: # Initialization
            self.w_p = 0.001 * np.random.randn(d)
            self.w_n = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            indices = np.random.choice(N, batch_size, replace=True)
            X_batch = X[indices]
            y_batch = y[indices]
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.  
            # Project for w_p and w_n
            self.w_p = np.maximum(self.w_p - learning_rate * (dw + self.alpha), 0)
            self.w_n = np.maximum(self.w_n + learning_rate * (dw - self.alpha), 0)
            self.b -= learning_rate * db
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))

    def predict(self, X):
        w = self.w_p - self.w_n
        return X @ w + self.b

    def loss(self, X_batch, y_batch):
        w = self.w_p - self.w_n
        return mse_loss_vectorized(w, self.b, X_batch, y_batch)

In [13]:
model = LassoProjectedGradientDescent(alpha=0.1)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 30542.466041
iteration 10000 / 200000: loss 4159.942572
iteration 20000 / 200000: loss 3114.031801
iteration 30000 / 200000: loss 3043.851649
iteration 40000 / 200000: loss 3085.578065
iteration 50000 / 200000: loss 3072.712256
iteration 60000 / 200000: loss 3195.069558
iteration 70000 / 200000: loss 3011.312356
iteration 80000 / 200000: loss 3253.377016
iteration 90000 / 200000: loss 2343.001680
iteration 100000 / 200000: loss 2669.085991
iteration 110000 / 200000: loss 2994.493969
iteration 120000 / 200000: loss 2720.715796
iteration 130000 / 200000: loss 2552.034858
iteration 140000 / 200000: loss 3163.505207
iteration 150000 / 200000: loss 2899.076148
iteration 160000 / 200000: loss 3260.625893
iteration 170000 / 200000: loss 2845.394382
iteration 180000 / 200000: loss 2718.240722
iteration 190000 / 200000: loss 2670.576645
MSE scikit-learn: 2912.52749260362
MSE Coordinate descent model : 2888.308548829491


In [14]:
model = LassoProjectedGradientDescent(alpha=2)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 28054.665165
iteration 10000 / 200000: loss 4558.809618
iteration 20000 / 200000: loss 3818.542854
iteration 30000 / 200000: loss 3951.195897
iteration 40000 / 200000: loss 3985.698993
iteration 50000 / 200000: loss 3671.736007
iteration 60000 / 200000: loss 4240.171333
iteration 70000 / 200000: loss 3675.603182
iteration 80000 / 200000: loss 3536.528882
iteration 90000 / 200000: loss 3314.968125
iteration 100000 / 200000: loss 4357.531125
iteration 110000 / 200000: loss 4112.809294
iteration 120000 / 200000: loss 3748.558600
iteration 130000 / 200000: loss 4308.228523
iteration 140000 / 200000: loss 4201.941164
iteration 150000 / 200000: loss 4513.988819
iteration 160000 / 200000: loss 3629.131582
iteration 170000 / 200000: loss 4182.665851
iteration 180000 / 200000: loss 3753.454720
iteration 190000 / 200000: loss 3748.851697
MSE scikit-learn: 5650.290772564547
MSE Coordinate descent model : 3810.9045199527236


# Lasso Coordinate Descent (no intercept)

In [15]:
class LassoCoordinateDescent():
    def __init__(self, alpha=0.1):
        self.w = None
        self.alpha = alpha

    def train(self, X, y, num_iters=1000):
        N, d = X.shape
        
        self.w = np.zeros(d)
        col_norms_sq = np.sum(X ** 2, axis=0)  
        
        for _ in range(num_iters):
            for j in range(d):
                r_j = y - X @ self.w + X[:, j] * self.w[j]
                rho_j = X[:, j] @ r_j
                if col_norms_sq[j] > 1e-10:
                    self.w[j] = soft_threshold(rho_j / col_norms_sq[j], self.alpha / col_norms_sq[j])

    def predict(self, X): 
        return X @ self.w

In [16]:
model = LassoCoordinateDescent(alpha=0.1)
model.train(X, y)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=False)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

MSE scikit-learn: 26057.124496145752
MSE Coordinate descent model : 26004.303657948607


In [17]:
model = LassoCoordinateDescent(alpha=2)
model.train(X, y)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=False)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

MSE scikit-learn: 28794.887776106683
MSE Coordinate descent model : 26006.422489676806
